In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.cluster import OPTICS
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA
import numpy as np
import pickle
import matplotlib.pyplot as plt
import os
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
from collections import Counter

## Caricamento dei files

In [ ]:
# Imposta il percorso della cartella contenente i file pkl
path = '/content/cold_heads_monitoring/Dataset'
arrays = []
data_list = []

# Itera su tutti i file pkl nella cartella e caricali in una lista
for filename in os.listdir(path):
    if filename.endswith('.pkl'):
        with open(os.path.join(path, filename), 'rb') as f:
            # Carica l'array NumPy dal file e seleziona solo il secondo canale
            arr = np.asarray(pickle.load(f))[:, 1:2, :]
            arrays.append(arr)
            data_list.append(os.path.splitext(filename)[0])

print(data_list)

In [ ]:
all_data = arrays
arrays = [np.array(arrays[6]),np.array(arrays[2]),np.array(arrays[4])]

In [ ]:
'''
# numero di righe e colonne del plot
n_rows = len(arrays)
n_cols = 3

# creazione del plot
fig, axs = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))

# loop sui segnali
for i, signal in enumerate(arrays):
    # loop sui canali
    for j in range(signal.shape[1]):
        # selezione del canale corrente
        current_channel = signal[:, j, :]
        # calcolo dell'istogramma
        hist, bins = np.histogram(current_channel.flatten(), bins=50)
        # plot dell'istogramma
        axs[i, j].hist(current_channel.flatten(), bins=50)
        # titolo del plot
        axs[i, j].set_title(f"Istogramma segnale {i+1} (canale {j+1})")

# spaziatura tra i plot
plt.subplots_adjust(hspace=0.5)

# visualizzazione del plot
plt.show()
'''

## Bilanciamento del dataset

In [ ]:
# Calcola la dimensione minima tra gli array nella lista
#min_len = min([arr.shape[0] for arr in arrays])
min_len = 2000

# Bilancia gli array nella lista
balanced_arrays = [arr[:min_len] for arr in arrays]

# Unisci gli array bilanciati in un unico array NumPy
data = np.concatenate(balanced_arrays, axis=0)

data = np.squeeze(data,1)
data.shape

In [ ]:
# Creazione dell'array con valori distinti per ciascun blocco
labels = np.empty(len(data),dtype=object)
left_index = 0
right_index = min_len

for i in range(len(data_list)):

  labels[left_index:right_index] = i
  left_index += min_len
  right_index += min_len

print(labels)

## Training DBSCAN

I parametri principali di DBSCAN sono i seguenti:

* eps: il raggio dell'intorno di ogni punto. È il parametro più importante dell'algoritmo perché determina la distanza massima tra i punti che possono far parte dello stesso cluster. Aumentando il valore di eps, aumentiamo la distanza massima tra i punti che possono essere raggruppati nello stesso cluster.

* min_samples: il numero minimo di punti che devono essere presenti all'interno del raggio di eps perché un punto possa far parte di un cluster. Aumentando il valore di min_samples, aumentiamo il numero di punti necessari per formare un cluster e quindi aumentiamo il numero di cluster compatti che si formano.

* metric: la funzione di distanza utilizzata per calcolare la distanza tra i punti. La distanza di default è la distanza euclidea, ma è possibile specificare altre metriche di distanza come la distanza di Manhattan, la distanza di Chebyshev, la distanza di Minkowski, ecc.

* algorithm: l'algoritmo utilizzato per il calcolo dei punti raggruppati. La scelta dell'algoritmo dipende soprattutto dalla dimensione dei dati. L'algoritmo di default è "auto", che sceglie l'algoritmo in base alla forma dei dati. Gli altri algoritmi disponibili sono "ball_tree", "kd_tree" e "brute".


In [ ]:
pca = PCA(n_components=2)
data = pca.fit_transform(data)

In [ ]:
# crea il modello di clustering con DBSCAN
dbscan = DBSCAN(eps=1.8, min_samples=5) #default 0.8
dbscan.fit(data)
clustering_labels = dbscan.labels_
print(clustering_labels)

In [ ]:
def distance(nuovo_campione,metric='chebyshev'):
  # Calcola il centroide di ciascun cluster
  cluster_centroids = []
  for label in np.unique(clustering_labels):
      cluster_points = data[clustering_labels == label]
      centroid = np.mean(cluster_points, axis=0)
      cluster_centroids.append(centroid)

  cluster_centroids = np.array(cluster_centroids)

  # Calcola la distanza tra il nuovo campione e i centroidi dei cluster
  distanze = pairwise_distances(nuovo_campione.reshape(1, -1), cluster_centroids, metric=metric)

  # Identifica il cluster più vicino
  cluster_più_vicino = np.argmin(distanze)
  return(cluster_più_vicino)

  print(f"Cluster più vicino al nuovo campione: {cluster_più_vicino}")

In [ ]:
# Visualizza i risultati del clustering
n_clusters = len(set(clustering_labels)) - (1 if -1 in clustering_labels else 0)
print("Numero di cluster trovati:", n_clusters)

# Visualizza i punti anomali (cluster con etichetta -1)
n_anomalies = list(clustering_labels).count(-1)
print("Numero di punti anomali trovati:", n_anomalies)

In [ ]:

# Creare un DataFrame per i dati e le etichette di clustering
df = pd.DataFrame({'X': data[:, 0], 'Y': data[:, 1], 'Cluster': clustering_labels})

# Plottiamo i cluster usando Seaborn
sns.scatterplot(data=df, x='X', y='Y', hue='Cluster', palette='viridis', alpha=0.5)
plt.show()

In [ ]:
# Crea una visualizzazione interattiva con Plotly
fig = px.scatter(df, x='X', y='Y', color='Cluster',
                 color_continuous_scale='viridis',
                 title='Cold Heads Clusters',
                 labels={'Cluster': 'Cluster'})

# Abilita il panning e lo zoom
fig.update_xaxes(fixedrange=False)
fig.update_yaxes(fixedrange=False)

# Mostra la visualizzazione interattiva
fig.show()

In [ ]:
num_indices_correct = np.sum(clustering_labels == labels)

# Calcolo della percentuale di indici correttamente classificati
total_indices = len(clustering_labels)
percentage_correct = (num_indices_correct / total_indices) * 100

print("Percentuale di indici correttamente classificati:", percentage_correct, "%")

In DBSCAN, un label di -1 viene assegnato ai punti che non appartengono a nessun cluster, ovvero vengono etichettati come "rumore". Questi punti non soddisfano i criteri di densità richiesti per far parte di un cluster e quindi vengono ignorati durante la creazione dei cluster.

La presenza di punti con label -1 può essere utile per identificare eventuali outliers o punti anomali nei dati. Tuttavia, è importante notare che la scelta dei parametri dell'algoritmo DBSCAN, come l'eps e il min_samples, può influire sulla quantità di punti etichettati come rumore. Inoltre, la scelta di come gestire tali punti dipende dalle specifiche esigenze dell'applicazione.

Gli assi del grafico rappresentano le prime due componenti principali (PC) dei vettori nella lista. Le componenti principali sono calcolate utilizzando l'analisi delle componenti principali (PCA), che è una tecnica di riduzione della dimensionalità utilizzata per visualizzare dati ad alta dimensionalità in un spazio bidimensionale o tridimensionale.

In altre parole, gli assi del grafico rappresentano le due direzioni principali che spiegano la maggior parte della varianza nei dati. La prima componente principale rappresenta la direzione lungo la quale i dati hanno la maggiore varianza, mentre la seconda componente principale rappresenta la direzione lungo la quale i dati hanno la seconda maggiore varianza, ed è ortogonale alla prima.

Tuttavia, è importante notare che la PCA può essere influenzata dalla scala dei dati e dalla presenza di outliers. Inoltre, la scelta delle componenti principali da utilizzare per il plot dipende dalle caratteristiche dei dati e dalle esigenze dell'applicazione. In alcuni casi, potrebbe essere necessario utilizzare altre tecniche di riduzione della dimensionalità o di visualizzazione dei dati.

#Inference

In [ ]:
new_data= all_data[4][:2000, 0, :]
pca = PCA(n_components=2)
new_data = pca.fit_transform(new_data)

In [ ]:
result = []
for elem in new_data:
  result.append(distance(elem))
count = Counter(result)
elemento_piu_ripetuto = count.most_common(1)[0][0]
print(elemento_piu_ripetuto)

In [ ]:
labels = dbscan.fit_predict(new_data)
count = Counter(labels)
elemento_piu_ripetuto = count.most_common(1)[0][0]
print(elemento_piu_ripetuto)